<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk13_mostlyai_seeded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# wk13: MostlyAI seeded reproducibility test

```
wk13_mostlyai_seeded.ipynb

PURPOSE. Settle whether MostlyAI's Random State parameter actually produces
reproducible training, and obtain a reproducible headline figure for Table 1.

DESIGN. Three training runs:
  Run A  Random State = 42
  Run B  Random State = 42   <- must be IDENTICAL to A if seeding works
  Run C  Random State = 7    <- must DIFFER from A and B

WHY TWO RUNS AT THE SAME SEED. One seeded run gives a number but cannot tell
you whether the number is reproducible, which is the entire question. A and B
are the actual test. C is the control: if A, B and C all match, the seed is
being ignored and something else is fixing the output.

OUTCOMES AND WHAT EACH MEANS
  A == B, C differs  -> seeding works. The 0.0723 spread across the four
                        unseeded runs was my omission. Report Run A as the
                        headline MostlyAI figure, with the unseeded range
                        reported alongside as a methodological observation.
  A != B             -> the Random State parameter does not fix training
                        output. Stronger finding: the tool documents
                        reproducibility it does not deliver. Report it, and
                        report MostlyAI as a mean with a range.
  A == B == C        -> the seed argument is being ignored entirely. Check
                        that the SDK accepted the parameter rather than
                        silently dropping an unrecognised keyword.



In [8]:
!pip install -q -U "mostlyai[local]" datasets 2>&1 | tail -3


In [9]:
import os, re, json, time, platform, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)

RANDOM_STATE = 42
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

MANIFEST = {
    "run_id": RUN_ID,
    "started": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "random_state_downstream": RANDOM_STATE,
    "purpose": "test whether MostlyAI Random State produces reproducible training",
}

RESULTS = []

def log_result(method, protocol, granularity, y_true, y_pred, notes=""):
    """Single point of truth. Nothing in this notebook is ever typed by hand
    into a table. This is the habit that caused the CTGAN and MostlyAI errors."""
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cold_label = -3 if granularity == 7 else -1
    cold = f1_score(y_true, y_pred, labels=[cold_label],
                    average="macro", zero_division=0)
    row = {"method": method, "protocol": protocol, "granularity": granularity,
           "macro_f1": round(float(macro), 4), "cold_f1": round(float(cold), 4),
           "run_id": RUN_ID, "notes": notes}
    RESULTS.append(row)
    print(f"  {method:<26} {protocol:<10} {granularity}-class  "
          f"macro_f1={macro:.4f}  cold_f1={cold:.4f}")
    return row

print("Run ID:", RUN_ID)


Run ID: 20260811_223118


In [10]:
from datasets import load_dataset

dataset  = load_dataset("kopetri/AutoTherm", "indoor")
train_df = dataset["train"].to_pandas()

train_df["participant_id"] = train_df["file_name"].apply(
    lambda f: re.search(r"participant_\d+", f).group())
train_df["Label_3class"] = train_df["Label"].apply(
    lambda x: -1 if x <= -2 else (0 if x <= 1 else 1))

TEST_PARTICIPANTS = ["participant_14", "participant_16", "participant_20"]
train_split = train_df[~train_df["participant_id"].isin(TEST_PARTICIPANTS)]
test_split  = train_df[ train_df["participant_id"].isin(TEST_PARTICIPANTS)]

# Comparability gates. If either fails, nothing below can be compared to wk10
# or wk12 and you should stop.
assert len(train_split) == 1_276_709, "train rows differ from wk10/wk12"
assert len(test_split)  ==   290_019, "test rows differ from wk10/wk12"

DROP_COLS = ["file_name", "Timestamp", "participant_id",
             "Air-Velocity", "Metabolic-Rate",
             "Nose", "Neck", "RShoulder", "RElbow",
             "LShoulder", "LElbow", "REye", "LEye", "REar", "LEar",
             "Emotion-Self", "Emotion-ML", "Label", "Label_3class"]

def prepare_features(df, target_col):
    X = df.drop(columns=[c for c in DROP_COLS if c in df.columns],
                errors="ignore").copy()
    if "Gender" in X.columns:
        X["Gender"] = LabelEncoder().fit_transform(X["Gender"].astype(str))
    return X.select_dtypes(include=[np.number]), df[target_col]

X_train_7, y_train_7 = prepare_features(train_split, "Label")
X_test_7,  y_test_7  = prepare_features(test_split,  "Label")
X_train_3, y_train_3 = prepare_features(train_split, "Label_3class")
X_test_3,  y_test_3  = prepare_features(test_split,  "Label_3class")

FEATURE_COLS = list(X_train_7.columns)
assert len(FEATURE_COLS) == 18 and "Label" not in FEATURE_COLS
print(f"Split and features match wk10/wk12. {len(FEATURE_COLS)} features.")


Split and features match wk10/wk12. 18 features.


In [11]:
# READ THE SIGNATURE BEFORE TRAINING. Do not skip this.
# Python silently accepts unknown keyword arguments in some wrapper APIs, so a
# misspelled parameter can look like it worked while doing nothing at all.
import inspect
from mostlyai.sdk import MostlyAI

mostly = MostlyAI(local=True)
print("mostly.train signature:")
print(inspect.signature(mostly.train))
print()
print(inspect.getdoc(mostly.train)[:3000])

# LOOK FOR: a parameter controlling random state / seed, and how config is
# passed. In many versions the model settings go inside a `config` dict rather
# than as a top-level argument, e.g.
#     mostly.train(config={"name": ..., "tables": [{"model_configuration":
#                  {"random_state": 42}}]})
# Correct SEED_KW and the train call in CELL 6 to match what you see above.


Initializing Synthetic Data SDK 6.1.1 in LOCAL mode 🏠

Connected to ]8;id=469231;file:///root/mostlyai\/root/]8;;\]8;id=962798;file:///root/mostlyai\mostlyai]8;;\ with 83 GB RAM, 12 CPUs, 1x NVIDIA A100-SXM4-40GB available

mostly.train signature:
(config: mostlyai.sdk.domain.GeneratorConfig | dict | None = None, data: pandas.core.frame.DataFrame | str | pathlib.Path | None = None, name: str | None = None, start: bool = True, wait: bool = True, progress_bar: bool = True) -> mostlyai.sdk.domain.Generator

Create a generator resource. Once trained, it will include the model as well as optionally a model report.

Note: A generator is initially being configured. That training job can be either launched immediately or later. One can check progress via `g.training.progress()`. Once the job has finished, the generator is available for use.

Args:
    config (GeneratorConfig | dict | None): The configuration parameters of the generator to be created. Either `config` or `data` must be provided.
    data (pd.DataFrame | str | Path | None): A single pandas DataFrame, or a path to a CSV or PARQUET file. Either `config` or `data` must be provided.
    name (str | None): Name of the generator.
    start (bool): Whether

In [12]:
from importlib.metadata import version
from mostlyai.sdk.domain import (GeneratorConfig, SourceTableConfig,
                                 ModelConfiguration, SyntheticDatasetConfig)

print("SDK version:", version("mostlyai"))

for cls in (GeneratorConfig, SourceTableConfig, ModelConfiguration,
            SyntheticDatasetConfig):
    hits = [n for n in cls.model_fields
            if any(k in n.lower() for k in ("random", "seed", "state"))]
    print(f"\n{cls.__name__}")
    print("  all fields :", list(cls.model_fields))
    print("  seed-like  :", hits or "NONE")

SDK version: 6.1.1

GeneratorConfig
  all fields : ['client', 'extra_key_values', 'name', 'description', 'random_state', 'tables', 'constraints']
  seed-like  : ['random_state']

SourceTableConfig
  all fields : ['client', 'extra_key_values', 'name', 'source_connector_id', 'location', 'data', 'tabular_model_configuration', 'language_model_configuration', 'primary_key', 'foreign_keys', 'columns']
  seed-like  : NONE

ModelConfiguration
  all fields : ['client', 'extra_key_values', 'model', 'max_sample_size', 'batch_size', 'gradient_accumulation_steps', 'max_training_time', 'max_epochs', 'max_sequence_window', 'enable_flexible_generation', 'value_protection', 'rare_category_replacement_method', 'differential_privacy', 'compute', 'enable_model_report']
  seed-like  : NONE

SyntheticDatasetConfig
  all fields : ['client', 'extra_key_values', 'generator_id', 'name', 'description', 'random_state', 'tables', 'delivery', 'compute']
  seed-like  : ['random_state']


In [13]:
sdv_drop = ["file_name", "Timestamp", "participant_id",
            "Air-Velocity", "Metabolic-Rate",
            "Nose", "Neck", "RShoulder", "RElbow",
            "LShoulder", "LElbow", "REye", "LEye", "REar", "LEar",
            "Emotion-Self", "Emotion-ML", "Label_3class"]

mostly_train = train_split.drop(columns=sdv_drop, errors="ignore").copy()
mostly_train["Gender"] = LabelEncoder().fit_transform(
    mostly_train["Gender"].astype(str))

SAMPLE_SIZE = 100_000
mostly_train_sample = (
    mostly_train.groupby("Label", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), int(SAMPLE_SIZE * len(x) / len(mostly_train))),
        random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

# Identical input to wk10 and wk12, so all seven runs pool legitimately.
assert len(mostly_train_sample) == 99_996, "sample differs from wk10/wk12"
print(f"Training sample: {len(mostly_train_sample):,} rows (matches wk10/wk12)")


Training sample: 99,996 rows (matches wk10/wk12)


In [14]:
def prep_synth(df, target_col):
    d = df.copy()
    if target_col == "Label_3class" and "Label_3class" not in d.columns:
        d["Label_3class"] = d["Label"].apply(
            lambda x: -1 if x <= -2 else (0 if x <= 1 else 1))
    if "Gender" in d.columns and d["Gender"].dtype == object:
        d["Gender"] = LabelEncoder().fit_transform(d["Gender"].astype(str))
    X = d.reindex(columns=FEATURE_COLS).apply(pd.to_numeric, errors="coerce")
    y = d[target_col]
    keep = X.notna().all(axis=1)
    return X[keep].reset_index(drop=True), y[keep].reset_index(drop=True)


def evaluate_synth(synth_df, tag):
    for gran, tcol, Xtr, ytr, Xte, yte in [
        (7, "Label",        X_train_7, y_train_7, X_test_7, y_test_7),
        (3, "Label_3class", X_train_3, y_train_3, X_test_3, y_test_3),
    ]:
        Xs, ys = prep_synth(synth_df, tcol)
        if ys.nunique() < 2:
            print(f"  [skip] {tag} {gran}-class: fewer than 2 classes")
            continue
        clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                     n_jobs=-1).fit(Xs, ys)
        log_result(tag, "TSTR", gran, yte, clf.predict(Xte))
        Xa = pd.concat([Xtr.reset_index(drop=True), Xs]).reset_index(drop=True)
        ya = pd.concat([ytr.reset_index(drop=True), ys]).reset_index(drop=True)
        clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                     n_jobs=-1).fit(Xa, ya)
        log_result(tag, "Augmented", gran, yte, clf.predict(Xte))


print("Functions defined. Now run cell 9.")

Functions defined. Now run cell 9.


In [15]:
SYNTH = {}

for tag, seed in [("A_seed42", 42), ("B_seed42", 42), ("C_seed7", 7)]:
    print(f"\n=== MostlyAI {tag} (random_state={seed}) ===")
    t0 = time.time()

    g = mostly.train(
        config={
            "name": f"autotherm_{RUN_ID}_{tag}",
            "random_state": seed,
            "tables": [{"name": "autotherm", "data": mostly_train_sample}],
        },
        start=True, wait=True,
    )
    print(f"  trained in {time.time()-t0:.0f}s")

    # Seed generation as well, so a difference between runs is attributable
    # to training rather than to sampling. Fall back to probe if generate
    # is unavailable in this mode.
    try:
        sd = mostly.generate(
            g,
            config={
                "name": f"synth_{RUN_ID}_{tag}",
                "random_state": seed,
                "tables": [{"name": "autotherm",
                            "configuration": {
                                "sample_size": len(mostly_train_sample)}}],
            },
        )
        synth = sd.data()
        gen_path = "generate (seeded)"
    except Exception as e:
        print(f"  generate() failed, falling back to probe: {e}")
        synth = mostly.probe(g, size=len(mostly_train_sample))
        gen_path = "probe (sampling NOT seeded)"

    print(f"  generation path: {gen_path}")
    SYNTH[tag] = synth.copy()
    evaluate_synth(synth, f"MostlyAI {tag}")


=== MostlyAI A_seed42 (random_state=42) ===


Created generator 70f9f501-9c67-47fd-8de0-4c7f2b306fd9

Started generator training

Output()

🎉 Your generator is ready! Use it to create synthetic data. Publish it so others can do the same.

...

Created synthetic dataset 08bc82ea-784f-4e62-a8c2-e32fa5ed64e8 with generator 63b98bf6-ab07-42be-8771-ca72e8b1e23b

Started synthetic dataset generation

Output()

🎉 Your synthetic dataset is ready! Use it to consume the generated data. Publish it so others can do the same.

  generation path: generate (seeded)
  MostlyAI C_seed7           TSTR       7-class  macro_f1=0.2344  cold_f1=0.0000
  MostlyAI C_seed7           Augmented  7-class  macro_f1=0.2811  cold_f1=0.0000
  MostlyAI C_seed7           TSTR       3-class  macro_f1=0.6583  cold_f1=0.6773
  MostlyAI C_seed7           Augmented  3-class  macro_f1=0.7408  cold_f1=0.7184


In [19]:
print("=" * 62)
print("REPRODUCIBILITY TEST")
print("=" * 62)

A, B, C = SYNTH["A_seed42"], SYNTH["B_seed42"], SYNTH["C_seed7"]

ab_identical = A.equals(B)
ac_identical = A.equals(C)

print(f"A vs B (both seed 42), synthetic data identical: {ab_identical}")
print(f"A vs C (seed 42 vs 7), synthetic data identical: {ac_identical}")
print()

res = pd.DataFrame(RESULTS)
print(res.pivot_table(index=["granularity", "protocol"],
                      columns="method", values="macro_f1").to_string())
print()

if ab_identical and not ac_identical:
    verdict = ("SEEDING WORKS. Same seed reproduces exactly, different seed differs. "
               "The 0.0723 spread across the four unseeded runs was an omission on my "
               "part, not a tool limitation. Report Run A as the headline MostlyAI "
               "figure, with the unseeded range alongside.")
elif not ab_identical:
    verdict = ("SEEDING DOES NOT FIX OUTPUT. Two runs at the same random_state produced "
               "different synthetic data. Report MostlyAI as a mean with a range and "
               "state this explicitly.")
else:
    verdict = ("ALL THREE IDENTICAL INCLUDING THE DIFFERENT SEED. The seed is probably "
               "not reaching the model. Re-check before concluding anything.")

print("VERDICT:", verdict)
MANIFEST["verdict"] = verdict
MANIFEST["A_equals_B"] = bool(ab_identical)
MANIFEST["A_equals_C"] = bool(ac_identical)
MANIFEST["seed_parameter"] = "GeneratorConfig.random_state + SyntheticDatasetConfig.random_state"

REPRODUCIBILITY TEST
A vs B (both seed 42), synthetic data identical: True
A vs C (seed 42 vs 7), synthetic data identical: False

method                 MostlyAI A_seed42  MostlyAI B_seed42  MostlyAI C_seed7
granularity protocol                                                         
3           Augmented             0.6965             0.6965            0.7408
            TSTR                  0.6610             0.6610            0.6583
7           Augmented             0.2938             0.2938            0.2811
            TSTR                  0.2419             0.2419            0.2344

VERDICT: SEEDING WORKS. Same seed reproduces exactly, different seed differs. The 0.0723 spread across the four unseeded runs was an omission on my part, not a tool limitation. Report Run A as the headline MostlyAI figure, with the unseeded range alongside.


In [20]:
# NOTE: this generator ID belongs to run A_seed42 above, created earlier in this same session. MostlyAI local generators are session-scoped, so on a fresh runtime this ID won't resolve and this cell will fail until you swap in the ID printed by that run.
g_A = mostly.generators.get("70f9f501-9c67-47fd-8de0-4c7f2b306fd9")
synth_probe = mostly.probe(g_A, size=len(mostly_train_sample))
evaluate_synth(synth_probe, "MostlyAI A_seed42 PROBE")

  MostlyAI A_seed42 PROBE    TSTR       7-class  macro_f1=0.2461  cold_f1=0.0000
  MostlyAI A_seed42 PROBE    Augmented  7-class  macro_f1=0.2791  cold_f1=0.0000
  MostlyAI A_seed42 PROBE    TSTR       3-class  macro_f1=0.6807  cold_f1=0.7032
  MostlyAI A_seed42 PROBE    Augmented  3-class  macro_f1=0.7033  cold_f1=0.6977


In [21]:
res.to_csv("results/wk13_mostlyai_seeded.csv", index=False)
MANIFEST["finished"] = datetime.now().isoformat(timespec="seconds")
MANIFEST["n_conditions_logged"] = len(res)
with open("results/wk13_manifest.json", "w") as f:
    json.dump(MANIFEST, f, indent=2)

from google.colab import files
files.download("results/wk13_mostlyai_seeded.csv")
files.download("results/wk13_manifest.json")
print("Saved and downloaded.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloaded.


In [22]:
# Patch the manifest and re-save with the probe run included.

MANIFEST["n_conditions_logged"] = len(RESULTS)
MANIFEST["finished"] = datetime.now().isoformat(timespec="seconds")

MANIFEST["verdict"] = (
    "SEEDING WORKS: A.equals(B) is True on the synthetic dataframes, and the "
    "training logs are identical epoch by epoch. A.equals(C) is False. The "
    "0.0723 macro F1 spread across four unseeded runs in wk10/wk12 was an "
    "omission, not a tool limitation."
)

MANIFEST["reproducibility_is_not_stability"] = (
    "Seed 42 beats the 7-class baseline (0.2938 vs 0.2858); seed 7 beats the "
    "3-class baseline (0.7408 vs 0.7163). Both reproducible. Mechanism: early "
    "stopping fired at epoch 26 for seed 42 and epoch 39 for seed 7, so the "
    "seed alters effective model capacity."
)

MANIFEST["probe_vs_generate"] = (
    "Identical generator (seed 42) evaluated via probe() and via generate() "
    "gives different results: 3-class Cold F1 0.7032 (probe) vs 0.6655 "
    "(generate), a difference of +0.0377. wk1-wk12 used probe(); this notebook "
    "used generate(). Results across notebooks are not strictly comparable."
)

MANIFEST["retraction_minority_amplification"] = (
    "The earlier claim that synthetic data improves 3-class Cold F1 where the "
    "class is learnable is RETRACTED. Pooling all 14 MostlyAI observations "
    "against the 0.7120 baseline: mean 0.7192, SD 0.0313, range 0.6655-0.7839, "
    "8 of 14 above baseline. Indistinguishable from baseline. The apparent gain "
    "in wk10/wk12 (7 of 8 above) was run-to-run variance."
)

MANIFEST["unaffected"] = (
    "7-class Cold F1 = 0.0000 in all 14 observations, across every seed, every "
    "generation path and every session."
)

MANIFEST["generator_ids"] = {
    "A_seed42": "70f9f501-9c67-47fd-8de0-4c7f2b306fd9",
    "B_seed42": "14e45194-e050-4aac-b9a6-fbbeb59eddd1",
    "C_seed7":  "63b98bf6-ab07-42be-8771-ca72e8b1e23b",
}

pd.DataFrame(RESULTS).to_csv("results/wk13_mostlyai_seeded.csv", index=False)
with open("results/wk13_manifest.json", "w") as f:
    json.dump(MANIFEST, f, indent=2)

print("Conditions logged:", len(RESULTS), "(expect 16: 3 runs x 4, plus probe x 4)")

from google.colab import files
files.download("results/wk13_mostlyai_seeded.csv")
files.download("results/wk13_manifest.json")

Conditions logged: 16 (expect 16: 3 runs x 4, plus probe x 4)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Results and verdict

**Run ID:** see MANIFEST. SDK 6.1.1. Generator IDs: A `70f9f501-9c67-47fd-8de0-4c7f2b306fd9`,
B `14e45194-e050-4aac-b9a6-fbbeb59eddd1`, C `63b98bf6-ab07-42be-8771-ca72e8b1e23b`.

### 1. Seeding works

`random_state` on `GeneratorConfig` produces exact reproducibility. Runs A and B
(both seed 42) are identical on all four metrics AND identical epoch by epoch in
the training log (val loss 22.7480, 23.0821, 23.1845, 23.1126, 22.8674, 22.9830).
Run C (seed 7) differs throughout.

The 0.0723 macro F1 spread observed across four unseeded runs in wk10 and wk12 was
therefore an omission on my part, not a limitation of the tool. Earlier claims in
this project that "MostlyAI exposes no seed" are retracted.

### 2. Reproducibility is not stability

| Condition | Seed 42 | Seed 7 | Gap | vs baseline |
|---|---|---|---|---|
| 7-class TSTR | 0.2419 | 0.2344 | 0.0075 | both below 0.2858 |
| 7-class Augmented | **0.2938** | 0.2811 | 0.0127 | seed 42 BEATS |
| 3-class TSTR | 0.6610 | 0.6583 | 0.0027 | both below 0.7163 |
| 3-class Augmented | 0.6965 | **0.7408** | 0.0443 | seed 7 BEATS |

Seed 42 beats the baseline at 7 classes. Seed 7 beats it at 3 classes. Both runs
are fully reproducible. The conclusion drawn still depends on an arbitrary seed.

Mechanism: seed 42 stopped at 26 epochs, seed 7 at 39. The seed alters the
validation loss trajectory, which alters when early stopping fires, which alters
effective model capacity. Seed 7 also reached a lower validation loss (22.32 vs
22.75) while performing worse at 7 classes, so generator loss does not predict
downstream utility.

### 3. `probe()` and `generate()` are not equivalent

Same generator (seed 42), two generation paths:

| Metric | generate | probe | Diff |
|---|---|---|---|
| 7-class TSTR | 0.2419 | 0.2461 | +0.0042 |
| 7-class Augmented | 0.2938 | 0.2791 | −0.0147 |
| 3-class TSTR | 0.6610 | 0.6807 | +0.0197 |
| 3-class Cold F1 TSTR | 0.6655 | 0.7032 | +0.0377 |

wk1–wk12 used `probe()`; this notebook used `generate()`. Results across those
notebooks are therefore not strictly comparable.

### 4. RETRACTION: the minority-class amplification claim

Earlier analysis in this project claimed that synthetic data improves the 3-class
Cold F1 where the class is learnable, based on 7 of 8 unseeded runs exceeding the
0.7120 baseline. Six further observations do not support it.

All MostlyAI 3-class Cold F1 observations, baseline 0.7120:

| Session | n | Mean | Range | Above baseline |
|---|---|---|---|---|
| wk10, unseeded, probe | 2 | 0.7390 | 0.7341–0.7440 | 2/2 |
| wk12, unseeded, probe | 6 | 0.7392 | 0.7072–0.7839 | 5/6 |
| wk13, seeded | 6 | 0.6926 | 0.6655–0.7184 | 1/6 |
| **All** | **14** | **0.7192** | **0.6655–0.7839** | **8/14** |

Spread 0.1184, SD 0.0313, 8 of 14 above baseline. MostlyAI's 3-class Cold F1 is
indistinguishable from the real-data baseline. The apparent gain in wk10 and wk12
was run-to-run variance.

Generation path explains part of the wk13 shortfall but not all of it: wk13 probe
runs (0.7032, 0.6977) still fall below the wk10/wk12 probe range. The residual is
unexplained.

### 5. What is unaffected

**7-class Cold F1 = 0.0000 in all 14 observations**, across every seed, every
generation path and every session. The central finding of this dissertation is
supported by 14 independent runs.

### Reporting decision

MostlyAI is reported as a mean with a stated range across 14 runs, never as a
point estimate. The seeded seed-42 `generate` run is cited as the reproducible
reference condition.
